# Training for audio features

In [4]:
from utils import import_data_science, load_music_dataset
from utils import data_transformation
from utils import artifacts_gen
import utils


import_data_science(globals())

In [5]:
raw = np.load('/mnt/g/artifacts/signals.npy', allow_pickle=True)

signals = raw[:,3]
signals.shape


one_hot_encoding = np.vectorize(data_transformation.one_hot_function)


labels = raw[:,1]

labels = one_hot_encoding(labels)
labels.shape

(5000,)

In [6]:
from IPython.display import clear_output

X = []
Y = []

for index, (signal, label) in enumerate(zip(signals, labels)):
    try:
        # print(f"{index / labels.shape[0] * 100}%")
        signal.shape[0] == 308700
        X.append(signal)
        Y.append(label)
        # result.append((features, label))
    except Exception as e:
        # print(f"couldn't extract features, {e}")
        pass
    # clear_output(wait=True)


print(len(X), len(Y))

X_train = []
y_train = []


for (signal, label) in zip(X, Y):
    try:
        # print(f"{index / labels.shape[0] * 100}%")
        for i in range(5,10,1):
            X_train.append(signal[i*22050:(i+1)*22050])
            y_train.append(label)
        # result.append((features, label))
    except Exception as e:
        print(f"couldn't extract features, {e}")
    # clear_output(wait=True)

print(len(X_train), len(y_train))

del X
del Y
del labels
del signals
del raw


4828 4828
24140 24140


In [6]:
frame_size = 4096 
hop_size = 512
fft_window = 256
signal_duration = 14

path = os.getcwd()

In [7]:
from IPython.display import clear_output

X = []
y = []

for index, (signal, label) in enumerate(zip(signals, labels)):
    try:
        print(f"{index / labels.shape[0] * 100}%")
        features = data_transformation.signal_to_features(signal)
        X.append(features)
        y.append(label)
        # result.append((features, label))
    except Exception as e:
        print(f"couldn't extract features, {e}")
    clear_output(wait=True)

print("feature transformation ready")


# Cleans out the np.nan
# Perform only once

print("cleaning transformation starting")

wrong_transformation = []

for index, sample in enumerate(X):
    try: 
        x = sample.shape
        pass
    except Exception as e:
        wrong_transformation.append(index)

print(f"impure signals count: {len(wrong_transformation)}")


good_x = []
good_y = []
for i, v in enumerate(X):
    if i not in wrong_transformation: 
        good_x.append(v)

for i, v in enumerate(y):
    if i not in wrong_transformation:
        good_y.append(v)

del X
del y
del signals
del raw

print(f"ready_x: {len(good_x)}")
print(f"ready_y: {len(good_y)}")



feature transformation ready
cleaning transformation starting
impure signals count: 172
ready_x: 4828
ready_y: 4828


In [7]:
def normalize_and_make_numpy(x, y):
    x = np.array(x).astype(np.float32)
    y = np.array(y).astype(np.float32)

    return x, y

x, y = normalize_and_make_numpy(X_train,y_train)

In [9]:
x.shape, y.shape

((24140, 22050), (24140,))

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset
import torch.nn.functional as F

In [21]:
class AudioDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # shape: (N, 1, 22050)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

from sklearn.model_selection import train_test_split

# x: shape (N, 3, 75, 20, 10), y: shape (N,)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.1, stratify=y, random_state=42
)


train_dataset = AudioDataset(x_train, y_train)
test_dataset = AudioDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [22]:
len(train_dataset), len(test_dataset)

(21726, 2414)

In [30]:
# 0.9 0.1 split 
# 45000 -> 512 -> 5. 55%
# 45000 -> 1024 -> 512 -> 5. 58%
# 45000 -> 1024 -> 512 -> 64 -> 5. 55.8%

In [23]:
# class MusicModel(nn.Module):
#     def __init__(self):
#         super(MusicModel, self).__init__()
#         self.linear_relu_stack = nn.Sequential(
#         nn.Linear(45000, 2048),
#         nn.ReLU(),
#         nn.Linear(2048, 512),
#         nn.ReLU(),
#         nn.Linear(512, 5),
#         )
# # 1024 - 48% acc
#     def forward(self, x):
#         logits = self.linear_relu_stack(x)
        
#         return logits


class MusicModel(nn.Module):
    def __init__(self):
        super(MusicModel, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv3d(3, 16, kernel_size=(3, 3, 3), padding=1),  # (C, T, H, W) → (16, ...)
            nn.ReLU(),
            nn.MaxPool3d((1, 2, 2)),  # reduce H, W
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3), padding=1),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),  # reduce T, H, W
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(32 * 37 * 5 * 2, 256),  # adjust based on actual output shape
            nn.ReLU(),
            nn.Linear(256, 5)  # assuming 5 classes
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5)
        self.pool1 = nn.MaxPool1d(4)
        self.bn1 = nn.BatchNorm1d(16)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=5)
        self.pool2 = nn.MaxPool1d(4)
        self.bn2 = nn.BatchNorm1d(32)

        self.conv3 = nn.Conv1d(32, 64, kernel_size=5)
        self.pool3 = nn.MaxPool1d(4)
        self.bn3 = nn.BatchNorm1d(64)

        self.flattened_size = self._get_flattened_size()

        self.fc1 = nn.Linear(self.flattened_size, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 5)

    def _get_flattened_size(self):
        # Dummy forward pass to calculate size after conv layers
        dummy = torch.zeros(1, 1, 22050)
        x = self.pool1(F.relu(self.conv1(dummy)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        return x.view(1, -1).size(1)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.bn1(x)

        x = self.pool2(F.relu(self.conv2(x)))
        x = self.bn2(x)

        x = self.pool3(F.relu(self.conv3(x)))
        x = self.bn3(x)

        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioCNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")

    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
    
    print(f"Validation Accuracy: {correct / total:.2%}")

Epoch 1: Loss = 1.5228
Validation Accuracy: 49.92%
Epoch 2: Loss = 1.2257
Validation Accuracy: 53.69%
Epoch 3: Loss = 1.1340
Validation Accuracy: 52.61%
Epoch 4: Loss = 1.0494
Validation Accuracy: 48.34%
Epoch 5: Loss = 0.9612
Validation Accuracy: 54.14%
Epoch 6: Loss = 0.8739
Validation Accuracy: 51.86%
Epoch 7: Loss = 0.7821
Validation Accuracy: 54.10%
Epoch 8: Loss = 0.7075
Validation Accuracy: 51.99%
Epoch 9: Loss = 0.6465
Validation Accuracy: 50.37%
Epoch 10: Loss = 0.5960
Validation Accuracy: 51.70%
Epoch 11: Loss = 0.5556
Validation Accuracy: 53.31%
Epoch 12: Loss = 0.5003
Validation Accuracy: 54.02%
Epoch 13: Loss = 0.4904
Validation Accuracy: 50.29%
Epoch 14: Loss = 0.4487
Validation Accuracy: 51.41%
Epoch 15: Loss = 0.4266
Validation Accuracy: 51.95%
Epoch 16: Loss = 0.4070
Validation Accuracy: 50.83%
Epoch 17: Loss = 0.3989
Validation Accuracy: 53.27%
Epoch 18: Loss = 0.3825
Validation Accuracy: 52.73%
Epoch 19: Loss = 0.3758
Validation Accuracy: 52.53%
Epoch 20: Loss = 0.35

In [36]:
waveform = torch.tensor(test_dataset[5][0], dtype=torch.float32).unsqueeze(0)  # shape (1, 1, 22050)

waveform = waveform.to(device)
model.eval()

with torch.no_grad():
    output = model(waveform)
    prediction = torch.argmax(output, dim=1).item()
    print(f"Predicted class: {prediction}")

test_dataset[5][1]

Predicted class: 0


/tmp/ipykernel_11955/1659763728.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(test_dataset[5][0], dtype=torch.float32).unsqueeze(0)  # shape (1, 1, 22050)


tensor(0)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MusicModel().to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min')

num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

    # Evaluate on test set
    model.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            test_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    test_acc = correct / total
    print(f"Test Acc: {test_acc:.4f}")
    scheduler.step(test_loss)

Epoch 1/15 | Loss: 266.2599 | Train Acc: 0.1926
Test Acc: 0.2039
Epoch 2/15 | Loss: 194.8379 | Train Acc: 0.1942
Test Acc: 0.2039
Epoch 3/15 | Loss: 194.8344 | Train Acc: 0.1965
Test Acc: 0.2029
Epoch 4/15 | Loss: 194.8837 | Train Acc: 0.1942
Test Acc: 0.2029
Epoch 5/15 | Loss: 194.8714 | Train Acc: 0.2007
Test Acc: 0.2039
Epoch 6/15 | Loss: 194.8588 | Train Acc: 0.2020
Test Acc: 0.1988
Epoch 7/15 | Loss: 194.8547 | Train Acc: 0.1955
Test Acc: 0.1988
Epoch 8/15 | Loss: 194.8653 | Train Acc: 0.2002
Test Acc: 0.1988
Epoch 9/15 | Loss: 194.8964 | Train Acc: 0.1862
Test Acc: 0.2039
Epoch 10/15 | Loss: 194.9370 | Train Acc: 0.1978
Test Acc: 0.2039
Epoch 11/15 | Loss: 194.8448 | Train Acc: 0.1999
Test Acc: 0.2039
Epoch 12/15 | Loss: 194.9044 | Train Acc: 0.2004
Test Acc: 0.1988
Epoch 13/15 | Loss: 194.8716 | Train Acc: 0.2002
Test Acc: 0.1988
Epoch 14/15 | Loss: 194.8977 | Train Acc: 0.1849
Test Acc: 0.1988
Epoch 15/15 | Loss: 194.8905 | Train Acc: 0.1916
Test Acc: 0.2029


In [68]:



output, actual = model(val_loader.dataset[1][0]), val_loader.dataset[1][1]
output, actual
torch.set_printoptions(precision=4, sci_mode=False)



props = F.softmax(output, dim=0) * 100
props, actual

(tensor([ 0.4582,  0.4070,  6.2674,  8.9495, 83.9179], grad_fn=<MulBackward0>),
 tensor(4.))

In [69]:
torch.save(model.state_dict(), '/mnt/g/models/0.pth')


In [70]:

upload_id = "003"
upload_dir = "/mnt/g/uploads/"
models_dir = "/mnt/g/models/"
artifacts_dir = "/mnt/g/song_artifacts/"
model_filename = "0.pth"
# TODO check and if not present make folders
# songslice
# videos
# power
# mels
# chroma
# mfcc


frame_size = 4096
hop_size = 512
fft_window = 512
signal_duration = 14

In [71]:
# Execution
filename = utils.find_file_by_upload_id(upload_id, upload_dir)
print(filename)

Upload couldn't be found
Upload couldn't be found
/mnt/g/uploads/Seek & Destroy (Remastered)-003.mp3


In [72]:
import librosa as li
import utils

# Execution
signal, sr = utils.extract_signal_for_inference(filename)

# Storing transformed signal
song_location = utils.generate_audio_from_frames(f"generated_track_{upload_id}", signal, sr, artifacts_dir)

# Frames indexes for generation of data
frames = utils.split_signal_to_frame_indexes(frame_size, hop_size, signal)

In [73]:
def extract_mels_and_generate_artifacts():
    mels = utils.extract_mels(frames, signal)
    utils.generate_mel_spec_images(upload_id, mels, artifacts_dir)
    utils.generate_video_from_mel_spec_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
    return mels

def extract_power_spectrograms_and_generate_artifacts():
    power_spectrograms = utils.extract_power_spectrograms(frames, signal)
    utils.generate_power_spectr_spec_images(upload_id, power_spectrograms, artifacts_dir)
    utils.generate_video_from_power_spetr_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
    return power_spectrograms

def extract_mfcc_and_generate_artifacts():
    mfcc = utils.extract_mfccs(frames, signal)
    utils.generate_mfcc_images(upload_id, mfcc, artifacts_dir)
    utils.generate_video_from_mfcc_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
    return mfcc

mels = extract_mels_and_generate_artifacts()
power_spectrograms = extract_power_spectrograms_and_generate_artifacts()
mfcc = extract_mfcc_and_generate_artifacts()

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [74]:
stacked = utils.stack_mfcc_power_spectro_mel_spectro(mfcc, mels, power_spectrograms)
stacked.shape

x = np.array(stacked).astype(np.float32)

flattened = torch.from_numpy(x).reshape(-1)

output = model(flattened)
percentages = F.softmax(output, dim=0) * 100

In [75]:
percentages

tensor([    0.0000,   100.0000,     0.0000,     0.0000,     0.0000],
       grad_fn=<MulBackward0>)